# Разбор метрики
## Формула Brier
$$Brier = \frac{1}{N} \sum_{i=1}^{N} (p_i - y_i)^2$$

Можно заметить, что при случайном выборе из двух классов, то есть p<sub>i</sub> = 0.5, Brier будет равен 0.5<sup>2</sup> = 0.25. Метрика идентична **MSE**, но для другого типа данных => ее нужно минимизировать. Переведем задачу в максимизацию.

Получаем формулу **1-Brier**
$$1-Brier = 1 - \frac{1}{N} \sum_{i=1}^{N} (p_i - y_i)^2$$

Которая при случайном выборе будет давать результат 1 - 0.25 = 0.75. Это означает, что если метрика получается ниже 0.75, то эта метрика работает хуже константного решения, поэтому приемлемый диапазон решения должен быть **[0.75, 1]**.

# Подход к решению
1. Сборка датасета неперевернутых картинок.
2. Аугментация и переворот картинок, чтобы сгенерировать класс 1.
3. Выбор и обучение модели (конечная модель **Convnext-Tiny**).
4. Проверка результатов на валидации.


# Импортируем все нужные библиотеки

In [52]:
import albumentations as A
from albumentations.pytorch import ToTensorV2
import os
import cv2
import torch
from torch.utils.data import Dataset
import numpy as np
import shutil
from tqdm import tqdm
from torch.utils.data import DataLoader
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.models as models
import matplotlib.pyplot as plt
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, accuracy_score, confusion_matrix
import csv

# Преобразования и аугментация
Так как задача сводится к классификации изображений текста и слов, то нужно выбирать широкое соотношение сторон.


Среднее соотношение ширины к высоте в тестовых данных находится между 4 и 5. После нескольких обучений модели, соотношение **4 к 1** показало наилучший результат, поэтому выбрано именно оно.

Также были протестированы : **128x640**, **64x320**, **256x1280**. 

Из размерности 64x320 ни одна модель не могла извлечь признаки, так как после сверток карта признаков была слишком мала.

**128x640** в сравнении с **224x896** показал существенный прирост метрики. Однако **224x896** в сравнении с **256x1280** не показал рост метрики, поэтому наиболее оптимальным разрешением стало **224x896**.

In [44]:
IMG_HEIGHT = 224
IMG_WIDTH = 896

train_transforms = A.Compose([
    #Фиксирую разрешение картинок для входа модели
    A.Resize(height=IMG_HEIGHT, width=IMG_WIDTH, interpolation=cv2.INTER_AREA, p=1.0),

    #Небольшой поворот, чтобы модель училась стабильнее
    A.Rotate(limit=7, border_mode=cv2.BORDER_REPLICATE, p=0.5),

    #Разная цветовая гамма
    A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5),
    A.RandomGamma(gamma_limit=(80, 120), p=0.3),
    A.CLAHE(clip_limit=2.0, tile_grid_size=(8, 8), p=0.2),
    A.HueSaturationValue(hue_shift_limit=10, sat_shift_limit=20, val_shift_limit=10, p=0.3),
    A.ToGray(p=0.1),

    #Шумы для имитации плохого качества фото или смазанных фото
    A.GaussianBlur(blur_limit=(3, 5), p=0.3),
    A.MotionBlur(blur_limit=3, p=0.2),
    A.GaussNoise(std_range=(0.1, 0.2), p=0.15),
    A.ISONoise(color_shift=(0.01, 0.05), intensity=(0.1, 0.5), p=0.2),
    A.ImageCompression(quality_range=(50, 90), p=0.3),

    #Стандартизация
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

#Валидация без шумов
val_transforms = A.Compose([
    A.Resize(height=IMG_HEIGHT, width=IMG_WIDTH, interpolation=cv2.INTER_AREA, p=1.0),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

test_transforms = val_transforms  

# Датасет
Из 1252 размеченного вручную изображения класса 0 поворотом на 180 градусов получаем класс 1. Сохраняем в такой структуре, чтобы класс изображения находился в названии папки.  

In [45]:
SRC_DIR = r'B:\Avito\Dataset'  
OUT_DIR = r'dataset_orientation'
os.makedirs(OUT_DIR, exist_ok=True)

#Чтение фото
def imread_unicode(path):
    img_array = np.fromfile(path, np.uint8)
    return cv2.imdecode(img_array, cv2.IMREAD_COLOR)

def imwrite_unicode(path, img):
    ext = os.path.splitext(path)[1]
    is_success, im_buf_arr = cv2.imencode(ext, img)
    if is_success:
        im_buf_arr.tofile(path)

if not os.path.exists(SRC_DIR):
    raise FileNotFoundError(f"Папка {SRC_DIR} не найдена")

files = [f for f in os.listdir(SRC_DIR) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]

#Получается, что 80% данных идет на обучение, 10% на валидацию и 10% на тестовые данные
train_files, temp = train_test_split(files, test_size=0.2, random_state=42)
val_files, test_files = train_test_split(temp, test_size=0.5, random_state=42)

def create_split(file_list, split):
    for cls in ['0', '1']:
        os.makedirs(os.path.join(OUT_DIR, split, cls), exist_ok=True)
        
    for fname in tqdm(file_list, desc=f"Processing {split}"):
        src_path = os.path.join(SRC_DIR, fname)
        img = imread_unicode(src_path)
        
        if img is None:
            continue
            
        out_path_0 = os.path.join(OUT_DIR, split, '0', fname)
        imwrite_unicode(out_path_0, img)
        
        img180 = cv2.rotate(img, cv2.ROTATE_180)
        out_path_1 = os.path.join(OUT_DIR, split, '1', fname)
        imwrite_unicode(out_path_1, img180)

create_split(train_files, 'train')
create_split(val_files, 'val')
create_split(test_files, 'test')

Processing test: 100%|██████████| 126/126 [00:00<00:00, 819.80it/s]


# Создание датасетов и даталоадеров

In [46]:
class OrientationDataset(Dataset):
    def __init__(self, root, transform=None):
        self.root = root
        self.transform = transform
        self.samples = []
        skipped = 0

        #Сбор путей к фотографиям разных классов
        for label in ['0', '1']:
            cls_dir = os.path.join(root, label)
            if not os.path.isdir(cls_dir):
                continue
            for fname in os.listdir(cls_dir):
                full_path = os.path.join(cls_dir, fname)
                if os.path.isfile(full_path) and os.path.getsize(full_path) > 0:
                    self.samples.append((full_path, int(label)))
                else:
                    skipped += 1

        print(f"[{root}] загружено : {len(self.samples)}, пропущено: {skipped}")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = imread_unicode(path)

        if img is None:
            return self.__getitem__((idx + 1) % len(self.samples))

        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        if self.transform:
            img = self.transform(image=img)['image']

        return img, label


train_dataset = OrientationDataset('dataset_orientation/train', train_transforms)
val_dataset   = OrientationDataset('dataset_orientation/val',   val_transforms)
test_dataset  = OrientationDataset('dataset_orientation/test',  test_transforms)

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True, num_workers=0)
val_loader   = DataLoader(val_dataset,   batch_size=4, shuffle=False, num_workers=0)
test_loader  = DataLoader(test_dataset,  batch_size=4, shuffle=False, num_workers=0)

[dataset_orientation/train] загружено : 2364, пропущено: 0
[dataset_orientation/val] загружено : 462, пропущено: 0
[dataset_orientation/test] загружено : 464, пропущено: 0


# Обучение
Протестированные модели : **собственные модели на Pytorch, Efficientnet-b0, Efficientnet-b2, Convnext-tiny, Resnet-50, Paddle**.

Собственная модель показала результат заметно хуже, чем **Efficientnet-b0** и **Resnet-50**, но работала крайне быстро.

Модель **Efficientnet-b0** и **Efficientnet-b2** не показали различий, возможно, из-за того, что модель **Efficientnet-b0** уже достаточно сложная для данной задачи.

**Convnext-tiny** работает быстрее **Efficientnet** и показывает результат гораздо лучше, 0.93 и 0.9 соответственно.

**Paddle OCR** тестировалась модель определения ориентации строк, однако она дала плохой результат в задаче.



In [ ]:
#Подсчет целевой метрики
def calculate_brier_score(y_true, y_pred_probs):
    return np.mean((y_pred_probs - y_true) ** 2)

def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0

    for images, labels in loader:
        images = images.to(device)
        labels = labels.float().to(device)

        optimizer.zero_grad()
        outputs = model(images).squeeze(-1)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)

    return running_loss / len(loader.dataset)


def validate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.float().to(device)

            outputs = model(images).squeeze(-1)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)

            probs = torch.sigmoid(outputs)
            all_preds.extend(probs.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    val_loss = running_loss / len(loader.dataset)
    brier_score = calculate_brier_score(np.array(all_labels), np.array(all_preds))
    competition_metric = 1 - brier_score

    return val_loss, competition_metric


def build_model(device):
    model = models.convnext_tiny(weights=models.ConvNeXt_Tiny_Weights.DEFAULT)

    #Отделяем классификатор в модели Convnext и вместо него добавляем линейный слой
    in_features = model.classifier[2].in_features
    model.classifier[2] = nn.Linear(in_features, 1)

    model = model.to(device)
    return model.to(device)


def train(model, train_loader, val_loader, device, best_model_path="best_model.pth"):
    #Количество эпох обучения только классификатора
    EPOCHS_STAGE_1 = 5
    #Количество эпох обучения всей модели
    EPOCHS_STAGE_2 = 15

    criterion = nn.BCEWithLogitsLoss()
    best_metric = -float("inf")

    print("Обучение только классификатора")
    for param in model.features.parameters():
        param.requires_grad = False

    for param in model.classifier.parameters():
        param.requires_grad = True

    #Добавляем небольшую регуляризацию
    optimizer = torch.optim.AdamW(
        model.classifier.parameters(),
        lr=1e-3,
        weight_decay=1e-4
    )

    for epoch in tqdm(range(EPOCHS_STAGE_1)):
        train_loss = train_one_epoch(model, train_loader, criterion, optimizer, device)
        val_loss, metric = validate(model, val_loader, criterion, device)
        print(
            f"Эпоха {epoch + 1}/{EPOCHS_STAGE_1} | "
            f"Потери обучения: {train_loss:.4f} | "
            f"Потери валидации: {val_loss:.4f} | "
            f"1-Brier: {metric:.4f}"
        )

    print("Обучение всей модели")
    for param in model.parameters():
        param.requires_grad = True

    #Добавляем сильную регуляризацию, чтобы обобщение было лучше
    optimizer = optim.AdamW(
    model.parameters(),
    lr=5e-5,
    weight_decay=0.05
)
    #Изменяем learning_rate
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS_STAGE_2)

    for epoch in tqdm(range(EPOCHS_STAGE_2)):
        train_loss = train_one_epoch(model, train_loader, criterion, optimizer, device)
        val_loss, metric = validate(model, val_loader, criterion, device)
        scheduler.step()

        print(
            f"Эпоха {epoch + 1}/{EPOCHS_STAGE_2} | "
            f"Потери обучнения: {train_loss:.4f} | "
            f"Потери валидации: {val_loss:.4f} | "
            f"1-Brier: {metric:.4f}"
        )

        if metric > best_metric:
            best_metric = metric
            torch.save(model.state_dict(), best_model_path)
            print(f"  -> Saved best model (1-Brier: {metric:.4f})")

    print(f"Лучший результат: {best_metric:.4f}")
    return best_metric


if __name__ == "__main__":
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Device: {device}")

    model = build_model(device)
    train(model, train_loader, val_loader, device)

Device: cuda
Обучение только классификатора


 20%|██        | 1/5 [00:59<03:56, 59.22s/it]

Эпоха 1/5 | Потери обучения: 0.7102 | Потери валидации: 0.6864 | 1-Brier: 0.7533


 40%|████      | 2/5 [01:58<02:57, 59.06s/it]

Эпоха 2/5 | Потери обучения: 0.6961 | Потери валидации: 0.6805 | 1-Brier: 0.7563


...

Эпоха 13/15 | Потери обучнения: 0.0055 | Потери валидации: 0.0406 | 1-Brier: 0.9926


 93%|█████████▎| 14/15 [44:22<03:06, 186.02s/it]

Эпоха 14/15 | Потери обучнения: 0.0072 | Потери валидации: 0.0367 | 1-Brier: 0.9922


100%|██████████| 15/15 [47:26<00:00, 189.75s/it]

Эпоха 15/15 | Потери обучнения: 0.0044 | Потери валидации: 0.0368 | 1-Brier: 0.9921
Лучший результат: 0.9934


# Проверка на тестовых данных

In [49]:
TEST_DIR = "dataset_orientation/test"
MODEL_PATH = "best_model.pth"
BATCH_SIZE = 8



def load_model(device):
    model = models.convnext_tiny(weights=models.ConvNeXt_Tiny_Weights.DEFAULT)

    in_features = model.classifier[2].in_features
    model.classifier[2] = nn.Linear(in_features, 1)
    model = model.to(device)
    #Загрузка модели
    model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
    model.eval()
    return model

@torch.no_grad()
def evaluate(model, loader, device):
    all_probs = []
    all_labels = []

    for images, labels in loader:
        images = images.to(device)
        logits = model(images).squeeze(-1)
        probs = torch.sigmoid(logits).cpu().numpy()

        all_probs.extend(probs)
        all_labels.extend(labels.numpy())

    return np.array(all_labels), np.array(all_probs)


def main():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Device: {device}")

    test_dataset = OrientationDataset(TEST_DIR, val_transforms)
    test_loader = DataLoader(
        test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0
    )

    model = load_model(device)
    y_true, y_prob = evaluate(model, test_loader, device)
    brier = calculate_brier_score(y_true, y_prob)
    metric_1_minus_brier = 1 - brier
    y_pred = (y_prob >= 0.5).astype(int)
    acc = accuracy_score(y_true, y_pred)

    try:
        auc = roc_auc_score(y_true, y_prob)
    except ValueError:
        auc = float("nan")

    print("\nРезультат на тесте")
    print(f"Всего примеров:  {len(y_true)}")
    print(f"Класс 0 : {int((y_true == 0).sum())}")
    print(f"Класс 1 : {int((y_true == 1).sum())}")
    print()
    print(f"Brier : {brier:.4f}")
    print(f"1 - Brier : {metric_1_minus_brier:.4f}")
    print(f"Accuracy : {acc:.4f}")
    print(f"ROC-AUC : {auc:.4f}")
    
    #Проверка уверенности модели
    print("\nРаспределение вероятностей класса 1:")
    for lo, hi in [(0.0, 0.1), (0.1, 0.3), (0.3, 0.7), (0.7, 0.9), (0.9, 1.0)]:
        mask = (y_prob >= lo) & (y_prob < hi) if hi < 1.0 else (y_prob >= lo) & (y_prob <= hi)
        n0 = int(((y_true == 0) & mask).sum())
        n1 = int(((y_true == 1) & mask).sum())
        print(f"[{lo:.1f}, {hi:.1f}): класс_0 = {n0:4d}, класс_1 = {n1:4d}")

    return metric_1_minus_brier

if __name__ == "__main__":
    main()

Device: cuda
[dataset_orientation/test] загружено : 464, пропущено: 0

Результат на тесте
Всего примеров:  464
Класс 0 : 232
Класс 1 : 232

Brier : 0.0057
1 - Brier : 0.9943
Accuracy : 0.9914
ROC-AUC : 0.9999

Распределение вероятностей класса 1:
[0.0, 0.1): класс_0 =  228, класс_1 =    0
[0.1, 0.3): класс_0 =    0, класс_1 =    0
[0.3, 0.7): класс_0 =    1, класс_1 =    0
[0.7, 0.9): класс_0 =    2, класс_1 =    2
[0.9, 1.0): класс_0 =    1, класс_1 =  230


In [ ]:
MODEL_PATH = "best_model.pth"
#Путь к распакованному архиву из задания
INPUT_DIR = r"B:\Avito\Test"         
OUTPUT_CSV = "submission.csv"
BATCH_SIZE = 8
NUM_WORKERS = 0                       

infer_transforms = A.Compose([
    A.Resize(IMG_HEIGHT, IMG_WIDTH, interpolation=cv2.INTER_AREA, p=1.0),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

#Класс для формирования даталоадера без разметки
class InferenceDataset(Dataset):
    def __init__(self, image_paths, transform):
        self.image_paths = image_paths
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        path = self.image_paths[idx]
        image_id = os.path.splitext(os.path.basename(path))[0]

        img = imread_unicode(path)
        if img is None:
            img = np.zeros((IMG_HEIGHT, IMG_WIDTH, 3), dtype=np.uint8)

        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        tensor = self.transform(image=img)["image"]
        return tensor, image_id

def collect_images(root_dir):
    exts = (".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff")
    paths = []
    for name in os.listdir(root_dir):
        full = os.path.join(root_dir, name)
        if os.path.isfile(full) and name.lower().endswith(exts):
            paths.append(full)
    return paths


@torch.no_grad()
def predict_all(model, loader, device):
    results = []
    for images, image_ids in tqdm(loader, desc="Predict"):
        images = images.to(device, non_blocking=True)
        logits = model(images).squeeze(-1)
        probs = torch.sigmoid(logits).cpu().numpy()
        for image_id, p in zip(image_ids, probs):
            results.append((image_id, float(p)))
    return results


def main():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Device: {device}")

    paths = collect_images(INPUT_DIR)
    print(f"Найдено файлов: {len(paths)}")
    if len(paths) == 0:
        raise RuntimeError(f"Не нашлось ни одной картинки в {INPUT_DIR}")

    #Сортировка по названию фото
    paths.sort(key=lambda p: os.path.splitext(os.path.basename(p))[0])

    dataset = InferenceDataset(paths, infer_transforms)
    loader = DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=(device.type == "cuda"),
    )

    model = load_model(device)
    results = predict_all(model, loader, device)

    #Сохранение в нужном формате
    with open(OUTPUT_CSV, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["image_id", "p_180"])
        for image_id, p in results:
            writer.writerow([image_id, f"{p:.6f}"])

    print(f"\n Сохранено {len(results)} строк в {OUTPUT_CSV}")

    #Характеристики для оценки уверенности модели
    probs = np.array([p for _, p in results])
    print(f"\nРаспределение p_180:")
    print(f"  min = {probs.min():.4f}")
    print(f"  median = {np.median(probs):.4f}")
    print(f"  mean = {probs.mean():.4f}")
    print(f"  max = {probs.max():.4f}")

    n_normal   = int((probs < 0.1).sum())
    n_flipped  = int((probs > 0.9).sum())
    n_uncertain = int(((probs >= 0.3) & (probs <= 0.7)).sum())

    print(f"\n p < 0.1 : {n_normal} ({100*n_normal/len(probs):.1f}%)")
    print(f"p > 0.9 : {n_flipped} ({100*n_flipped/len(probs):.1f}%)")
    print(f"0.3–0.7 : {n_uncertain} ({100*n_uncertain/len(probs):.1f}%)")


if __name__ == "__main__":
    main()

Device: cuda
Найдено файлов: 20000


Predict: 100%|██████████| 2500/2500 [04:52<00:00,  8.55it/s]


 Сохранено 20000 строк в submission.csv

Распределение p_180:
  min = 0.0000
  median = 0.6804
  mean = 0.5131
  max = 1.0000

 p < 0.1 : 8948 (44.7%)
p > 0.9 : 9465 (47.3%)
0.3–0.7 : 607 (3.0%)
